# Generate Sample Data

**Run this notebook after `00_Lab_Environment_Setup`.**

This notebook creates sample datasets for all 6 labs:

| Dataset | Format | Records | Used In |
|---------|--------|---------|----------|
| AXA Entities | JSON | 18 | Reference data (all labs) |
| Reinsurance Contracts | CSV | 500 | Lab 02 - PySpark Transformation |
| Insurance Claims | JSON | 1,000 | Lab 03 - Delta Lake |
| Placements (initial) | JSON | 300 | Lab 04 - Bronze Layer (batch) |
| Placements (incremental) | JSON | 80 | Lab 04 - Bronze Layer (Auto Loader) |

**Day 1 data** is written to Unity Catalog Volumes.  
**Day 2 data** is written to ADLS Gen2 (if configured).

In [ ]:
# ============================================================
# CONFIGURATION - Must match 00_Lab_Environment_Setup
# ============================================================

CATALOG_NAME = dbutils.secrets.get(scope="training", key="catalog-name")
SCHEMA_ASSETS = "lab_assets"
VOLUME_NAME = "training_volume"
VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_ASSETS}/{VOLUME_NAME}"
RAW_DATA_PATH = f"{VOLUME_PATH}/raw_data"

# ADLS paths (UC External Location handles auth — no keys needed)
STORAGE_ACCOUNT = dbutils.secrets.get(scope="training", key="storage-account-name")
CONTAINER = dbutils.secrets.get(scope="training", key="container")
WS_KEY = dbutils.secrets.get(scope="training", key="ws-key")
ADLS_ENABLED = True
BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{WS_KEY}"
ADLS_RAW_PATH = f"{BASE_PATH}/raw"

print(f"Volume Path:   {VOLUME_PATH}")
print(f"ADLS Raw Path: {ADLS_RAW_PATH}")

In [ ]:
# Imports
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import (
    col, lit, rand, round as spark_round, when,
    date_add, concat, lpad, array, element_at, floor
)
import uuid

print("Libraries imported.")

---

## 1. AXA Entities (Reference Data)

Reference table for AXA Group companies worldwide. Used for joins in all labs.

In [ ]:
entities_data = [
    ("AXA-FR-001", "AXA France", "Insurance", "Europe", "France", "Paris", "75008", "contact@axa.fr", "+33 1 40 75 57 00"),
    ("AXA-FR-002", "AXA France IARD", "Insurance", "Europe", "France", "Nanterre", "92000", "iard@axa.fr", "+33 1 47 74 10 00"),
    ("AXA-DE-001", "AXA Germany", "Insurance", "Europe", "Germany", "Cologne", "50672", "kontakt@axa.de", "+49 221 148 0"),
    ("AXA-UK-001", "AXA UK", "Insurance", "Europe", "United Kingdom", "London", "EC3M 7BA", "info@axa-uk.co.uk", "+44 20 7283 7171"),
    ("AXA-BE-001", "AXA Belgium", "Insurance", "Europe", "Belgium", "Brussels", "1170", "info@axa.be", "+32 2 678 61 11"),
    ("AXA-CH-001", "AXA Switzerland", "Insurance", "Europe", "Switzerland", "Winterthur", "8400", "info@axa.ch", "+41 58 215 21 21"),
    ("AXA-IT-001", "AXA Italy", "Insurance", "Europe", "Italy", "Milan", "20149", "info@axa.it", "+39 02 4809 1"),
    ("AXA-ES-001", "AXA Spain", "Insurance", "Europe", "Spain", "Madrid", "28034", "info@axa.es", "+34 91 538 50 00"),
    ("AXA-US-001", "AXA XL US", "Reinsurance", "Americas", "United States", "Stamford", "06902", "info@axaxl.com", "+1 203 964 5200"),
    ("AXA-US-002", "AXA Equitable", "Life Insurance", "Americas", "United States", "New York", "10104", "info@equitable.com", "+1 212 554 1234"),
    ("AXA-MX-001", "AXA Mexico", "Insurance", "Americas", "Mexico", "Mexico City", "11000", "info@axa.mx", "+52 55 5169 1000"),
    ("AXA-BR-001", "AXA Brazil", "Insurance", "Americas", "Brazil", "Sao Paulo", "01452-000", "info@axa.com.br", "+55 11 3030 1234"),
    ("AXA-JP-001", "AXA Japan", "Insurance", "Asia Pacific", "Japan", "Tokyo", "108-8020", "info@axa.co.jp", "+81 3 6737 7777"),
    ("AXA-HK-001", "AXA Hong Kong", "Insurance", "Asia Pacific", "Hong Kong", "Hong Kong", "999077", "info@axa.com.hk", "+852 2519 1111"),
    ("AXA-SG-001", "AXA Singapore", "Insurance", "Asia Pacific", "Singapore", "Singapore", "048583", "info@axa.com.sg", "+65 6880 4888"),
    ("AXA-XL-001", "AXA XL Reinsurance", "Reinsurance", "Global", "Bermuda", "Hamilton", "HM11", "info@axaxl.com", "+1 441 292 8515"),
    ("AXA-IM-001", "AXA Investment Managers", "Asset Management", "Global", "France", "Paris", "75008", "info@axa-im.com", "+33 1 44 45 70 00"),
    ("AXA-GO-001", "AXA Global Operations", "Operations", "Global", "France", "Paris", "75008", "global.ops@axa.com", "+33 1 40 75 48 68"),
]

entities_schema = StructType([
    StructField("entity_id", StringType(), False),
    StructField("entity_name", StringType(), False),
    StructField("entity_type", StringType(), False),
    StructField("region", StringType(), False),
    StructField("country", StringType(), False),
    StructField("address", StructType([
        StructField("city", StringType(), False),
        StructField("postal_code", StringType(), False),
    ]), False),
    StructField("contact", StructType([
        StructField("email", StringType(), False),
        StructField("phone", StringType(), False),
    ]), False),
])

# Build rows with nested structs
from pyspark.sql import Row

entities_rows = [
    Row(
        entity_id=r[0], entity_name=r[1], entity_type=r[2],
        region=r[3], country=r[4],
        address=Row(city=r[5], postal_code=r[6]),
        contact=Row(email=r[7], phone=r[8])
    )
    for r in entities_data
]

df_entities = spark.createDataFrame(entities_rows, entities_schema)

# Save to Unity Catalog Volume (Day 1)
entities_vol_path = f"{RAW_DATA_PATH}/axa_entities"
df_entities.write.mode("overwrite").json(entities_vol_path)
print(f"Entities: {df_entities.count()} records -> {entities_vol_path}")

# Save to ADLS (Day 2)
if ADLS_ENABLED:
    entities_adls_path = f"{ADLS_RAW_PATH}/entities"
    df_entities.write.mode("overwrite").json(entities_adls_path)
    print(f"Entities: {df_entities.count()} records -> {entities_adls_path}")

display(df_entities)

---

## 2. Reinsurance Contracts (Lab 02 - PySpark Transformation)

500 simulated reinsurance contracts across entities and reinsurers.

In [ ]:
entity_ids = [row.entity_id for row in df_entities.collect()]
reinsurer_ids = ["SWISS-RE-001", "MUNICH-RE-001", "HANNOVER-RE-001", "SCOR-001", "LLOYDS-001"]
contract_types = ["QUOTA_SHARE", "EXCESS_OF_LOSS", "STOP_LOSS", "FACULTATIVE"]
currencies = ["EUR", "USD", "GBP", "CHF"]
statuses = ["ACTIVE", "ACTIVE", "ACTIVE", "EXPIRED", "PENDING"]

NUM_CONTRACTS = 500
df_base = spark.range(1, NUM_CONTRACTS + 1).withColumnRenamed("id", "row_num")

df_contracts = df_base \
    .withColumn("contract_id", concat(lit("RC-"), lpad(col("row_num").cast("string"), 6, "0"))) \
    .withColumn("axa_entity_id", element_at(array(*[lit(x) for x in entity_ids]), (floor(rand() * len(entity_ids)) + 1).cast("int"))) \
    .withColumn("reinsurer_id", element_at(array(*[lit(x) for x in reinsurer_ids]), (floor(rand() * len(reinsurer_ids)) + 1).cast("int"))) \
    .withColumn("contract_type", element_at(array(*[lit(x) for x in contract_types]), (floor(rand() * len(contract_types)) + 1).cast("int"))) \
    .withColumn("effective_date", date_add(lit("2022-01-01"), (rand() * 730).cast("int"))) \
    .withColumn("expiry_date", date_add(col("effective_date"), (rand() * 365 + 365).cast("int"))) \
    .withColumn("premium_amount", spark_round(rand() * 9000000 + 1000000, 2)) \
    .withColumn("cession_rate", spark_round(rand() * 0.4 + 0.1, 4)) \
    .withColumn("currency", element_at(array(*[lit(x) for x in currencies]), (floor(rand() * len(currencies)) + 1).cast("int"))) \
    .withColumn("status", element_at(array(*[lit(x) for x in statuses]), (floor(rand() * len(statuses)) + 1).cast("int"))) \
    .select(
        "contract_id", "axa_entity_id", "reinsurer_id", "contract_type",
        col("effective_date").cast("string"), col("expiry_date").cast("string"),
        "premium_amount", "cession_rate", "currency", "status"
    )

contracts_path = f"{RAW_DATA_PATH}/reinsurance_contracts"
df_contracts.write.mode("overwrite").option("header", "true").csv(contracts_path)

print(f"Contracts: {df_contracts.count()} records -> {contracts_path}")
display(df_contracts.limit(5))

---

## 3. Insurance Claims (Lab 03 - Delta Lake)

1,000 simulated insurance claims across different entities and claim types.

In [ ]:
claim_types = ["PROPERTY", "LIABILITY", "AUTO", "HEALTH", "MARINE", "AVIATION"]
claim_statuses = ["OPEN", "CLOSED", "PENDING", "UNDER_REVIEW"]

NUM_CLAIMS = 1000
df_claims_base = spark.range(1, NUM_CLAIMS + 1).withColumnRenamed("id", "row_num")

df_claims = df_claims_base \
    .withColumn("claim_id", concat(lit("CLM-"), lpad(col("row_num").cast("string"), 8, "0"))) \
    .withColumn("policy_id", concat(lit("POL-"), lpad((col("row_num") % 200 + 1).cast("string"), 6, "0"))) \
    .withColumn("axa_entity_id", element_at(array(*[lit(x) for x in entity_ids]), (floor(rand() * len(entity_ids)) + 1).cast("int"))) \
    .withColumn("claim_date", date_add(lit("2023-01-01"), (rand() * 400).cast("int"))) \
    .withColumn("claim_amount", spark_round(rand() * 490000 + 10000, 2)) \
    .withColumn("claim_type", element_at(array(*[lit(x) for x in claim_types]), (floor(rand() * len(claim_types)) + 1).cast("int"))) \
    .withColumn("claim_status", element_at(array(*[lit(x) for x in claim_statuses]), (floor(rand() * len(claim_statuses)) + 1).cast("int"))) \
    .withColumn("adjuster_id", concat(lit("ADJ-"), lpad((floor(rand() * 50) + 1).cast("string"), 3, "0"))) \
    .select(
        "claim_id", "policy_id", "axa_entity_id",
        col("claim_date").cast("timestamp").alias("claim_date"),
        "claim_amount", "claim_type", "claim_status", "adjuster_id"
    )

claims_path = f"{RAW_DATA_PATH}/claims"
df_claims.write.mode("overwrite").json(claims_path)

print(f"Claims: {df_claims.count()} records -> {claims_path}")
display(df_claims.limit(5))

---

## 4. Investment Placements (Lab 04/05/06 - E2E Pipeline)

Investment placement data written to ADLS Gen2 for the Day 2 E2E pipeline labs.

- **Initial batch** (300 records): For batch ingestion in Bronze Layer
- **Incremental batches** (50 + 30 records): For Auto Loader demo

In [ ]:
asset_classes = ["EQUITY", "BOND", "REAL_ESTATE", "ALTERNATIVE", "CASH", "DERIVATIVES"]

def generate_placements(start_id, count, start_date, date_range_days):
    """Generate placement data with configurable parameters.
    Includes ~5% nulls and ~3% negative values for data quality exercises."""
    df_base = spark.range(start_id, start_id + count).withColumnRenamed("id", "row_num")
    return df_base \
        .withColumn("placement_id", concat(lit("PLC-"), lpad(col("row_num").cast("string"), 6, "0"))) \
        .withColumn("axa_entity_id", element_at(array(*[lit(x) for x in entity_ids]), (floor(rand() * len(entity_ids)) + 1).cast("int"))) \
        .withColumn("asset_class", element_at(array(*[lit(x) for x in asset_classes]), (floor(rand() * len(asset_classes)) + 1).cast("int"))) \
        .withColumn("instrument_id", concat(lit("ISIN-"), lpad((col("row_num") % 100 + 1).cast("string"), 4, "0"))) \
        .withColumn("_raw_market_value", spark_round(rand() * 9000000 + 1000000, 2)) \
        .withColumn("_quality_rand", rand()) \
        .withColumn("market_value",
            when(col("_quality_rand") < 0.05, lit(None).cast("double"))
            .when(col("_quality_rand") < 0.08, spark_round(rand() * -500000, 2))
            .otherwise(col("_raw_market_value"))
        ) \
        .withColumn("book_value",
            when(col("_quality_rand") > 0.93, lit(None).cast("double"))
            .when(col("_quality_rand") > 0.90, spark_round(rand() * -200000, 2))
            .otherwise(spark_round(col("_raw_market_value") * (rand() * 0.3 + 0.85), 2))
        ) \
        .withColumn("currency", element_at(array(*[lit(x) for x in currencies]), (floor(rand() * len(currencies)) + 1).cast("int"))) \
        .withColumn("valuation_date",
            when(rand() < 0.03, lit(None).cast("string"))
            .otherwise(date_add(lit(start_date), (rand() * date_range_days).cast("int")).cast("string"))
        ) \
        .select(
            "placement_id", "axa_entity_id", "asset_class", "instrument_id",
            "market_value", "book_value", "currency", "valuation_date"
        )

print("Placement generator ready (with ~5% nulls + ~3% negatives for quality exercises).")

In [ ]:
# Generate initial batch (300 records)
df_placements = generate_placements(1, 300, "2024-01-01", 365)

# Save to Volume (backup)
placements_vol_path = f"{RAW_DATA_PATH}/placements"
df_placements.write.mode("overwrite").json(placements_vol_path)
print(f"Placements (initial): {df_placements.count()} records -> {placements_vol_path}")

# Save to ADLS (for Day 2 labs)
if ADLS_ENABLED:
    adls_placements_path = f"{ADLS_RAW_PATH}/placements"
    df_placements.write.mode("overwrite").json(adls_placements_path)
    print(f"Placements (initial): {df_placements.count()} records -> {adls_placements_path}")

display(df_placements.limit(5))

In [ ]:
# Generate incremental batches for Auto Loader
df_inc1 = generate_placements(301, 50, "2024-06-01", 90)
df_inc2 = generate_placements(351, 30, "2024-09-01", 60)

# Save to Volume (fixed batch names for idempotency)
inc_vol_path = f"{RAW_DATA_PATH}/placements_incremental"

# Clear existing incremental data to avoid duplicates on re-run
try:
    dbutils.fs.rm(inc_vol_path, recurse=True)
except:
    pass
dbutils.fs.mkdirs(inc_vol_path)

df_inc1.coalesce(1).write.mode("overwrite").json(f"{inc_vol_path}/batch_1")
df_inc2.coalesce(1).write.mode("overwrite").json(f"{inc_vol_path}/batch_2")
print(f"Incremental batch 1: {df_inc1.count()} records")
print(f"Incremental batch 2: {df_inc2.count()} records")

# Save to ADLS
if ADLS_ENABLED:
    adls_inc_path = f"{ADLS_RAW_PATH}/placements_incremental"
    try:
        dbutils.fs.rm(adls_inc_path, recurse=True)
    except:
        pass
    df_inc1.coalesce(1).write.mode("overwrite").json(f"{adls_inc_path}/batch_1")
    df_inc2.coalesce(1).write.mode("overwrite").json(f"{adls_inc_path}/batch_2")
    print(f"ADLS incremental batches saved to {adls_inc_path}")

---

## 5. Verification Summary

In [ ]:
print("=" * 70)
print("DATA GENERATION SUMMARY")
print("=" * 70)

datasets = [
    ("Entities (Reference)", f"{RAW_DATA_PATH}/axa_entities", "JSON", "All labs"),
    ("Contracts (PySpark Lab)", f"{RAW_DATA_PATH}/reinsurance_contracts", "CSV", "Lab 02"),
    ("Claims (Delta Lake Lab)", f"{RAW_DATA_PATH}/claims", "JSON", "Lab 03"),
    ("Placements Initial", f"{RAW_DATA_PATH}/placements", "JSON", "Lab 04 (batch)"),
    ("Placements Incremental", f"{RAW_DATA_PATH}/placements_incremental", "JSON", "Lab 04 (autoloader)"),
]

for name, path, fmt, lab in datasets:
    try:
        files = dbutils.fs.ls(path)
        file_count = len([f for f in files if not f.name.startswith("_")])
        print(f"\n  {name}")
        print(f"    Path:   {path}")
        print(f"    Format: {fmt} | Items: {file_count} | Lab: {lab}")
    except Exception as e:
        print(f"\n  {name}: ERROR - {e}")

if ADLS_ENABLED:
    print(f"\n  --- ADLS Gen2 Data ---")
    for name, subpath in [("Entities", "entities"), ("Placements", "placements"), ("Incremental", "placements_incremental")]:
        try:
            files = dbutils.fs.ls(f"{ADLS_RAW_PATH}/{subpath}")
            print(f"  {name}: {len(files)} items at {ADLS_RAW_PATH}/{subpath}")
        except Exception as e:
            print(f"  {name}: ERROR - {e}")

print("\n" + "=" * 70)
print("DATA GENERATION COMPLETE")
print("=" * 70)
print("\nReady for lab notebooks.")

---

## Data Paths Reference

### Day 1 Labs (Unity Catalog Volumes)

| Dataset | Path | Format |
|---------|------|--------|
| Entities | `/Volumes/axa_training/lab_assets/training_volume/raw_data/axa_entities` | JSON |
| Contracts | `/Volumes/axa_training/lab_assets/training_volume/raw_data/reinsurance_contracts` | CSV |
| Claims | `/Volumes/axa_training/lab_assets/training_volume/raw_data/claims` | JSON |

### Day 2 Labs (ADLS Gen2)

| Dataset | Path Pattern | Format |
|---------|-------------|--------|
| Entities | `{BASE_PATH}/raw/entities` | JSON |
| Placements (batch) | `{BASE_PATH}/raw/placements` | JSON |
| Placements (incremental) | `{BASE_PATH}/raw/placements_incremental` | JSON |

---

## Next Steps

Proceed to the lab notebooks:

**Day 1:**
1. `jour1/01_Getting_Started_with_Databricks.ipynb`
2. `jour1/02_PySpark_Transformation_Lab.ipynb`
3. `jour1/03_Delta_Lake_Lab.ipynb`

**Day 2:**
4. `jour2/04_Bronze_Layer.ipynb`
5. `jour2/05_Silver_Layer.ipynb`
6. `jour2/06_Gold_Layer.ipynb`